# Event-fitting model gallery

## Available models, fitted parameters, and labeled curve features

This notebook is a visual guide to the event models registered in `Model_Calibration/event_models.py`. It uses the repository's real model functions and parameter order—no duplicate model equations are defined here.

The notebook answers three practical questions:

1. **Which fitting models are available?**
2. **Which parameters belong to each model?**
3. **Which visible feature of a response does each parameter control?**

> The curves below are illustrative model responses, not fits to experimental data. Amplitudes are normalized only in the overview gallery so kinetic shapes can be compared fairly.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt

# Locate the repository whether Jupyter starts at the repository root or here.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "Model_Calibration" / "event_models.py").exists():
    REPO_ROOT = Path("../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from Model_Calibration.event_models import get_event_model

plt.rcParams.update({
    "figure.figsize": (12, 4),
    "font.size": 10.5,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

SAVE_FIGURES = False
DOC_DIR = REPO_ROOT / "docs" / "model_fitting"
FIGURE_DIR = DOC_DIR / "figures"

def finish_figure(fig, stem):
    fig.tight_layout()
    if SAVE_FIGURES:
        FIGURE_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(FIGURE_DIR / f"{stem}.png", dpi=300, bbox_inches="tight")
        fig.savefig(FIGURE_DIR / f"{stem}.svg", bbox_inches="tight")
    plt.show()

print(f"Using model registry: {REPO_ROOT / 'Model_Calibration' / 'event_models.py'}")

---
## 1. Model catalogue

The gallery is grouped by conceptual complexity. The categories are organizational only; every model is obtained from the same `get_event_model()` registry.

In [ ]:
MODEL_GROUPS = {
    "Classical phenomenological models": [
        "single_exp", "double_exp", "alpha", "gamma", "bilinear", "cooperative",
    ],
    "iGluSnFR and kinetic models": [
        "two_step_binding", "binding_kinetics", "iglusnfr", "iglusnfr_tri",
    ],
    "Composite and heterogeneous models": [
        "two_component", "desensitization", "coop_plus_linear",
        "diffusion_clearance", "double_cooperative", "hetero_coop", "two_comp_coop",
    ],
}

# Representative values in the exact order declared by each registry spec.
EXAMPLE_PARAMS = {
    "single_exp":          [1.00, 0.025, 5.0],
    "double_exp":          [1.00, 0.002, 0.025, 5.0],
    "alpha":               [1.00, 0.012, 5.0],
    "gamma":               [1.00, 2.0, 0.009, 5.0],
    "bilinear":            [1.00, 3.0, 25.0, 5.0],
    "cooperative":         [1.00, 0.004, 0.030, 2.0, 5.0],
    "two_step_binding":    [1.00, 0.0005, 0.003, 0.045, 5.0],
    "binding_kinetics":    [1.00, 250.0, 40.0, 0.030, 5.0],
    "iglusnfr":            [1.00, 0.002, 0.007, 0.030, 0.65, 5.0],
    "iglusnfr_tri":        [1.00, 0.0015, 0.005, 0.018, 0.060, 0.50, 0.30, 5.0],
    "two_component":       [0.65, 0.002, 0.012, 0.35, 0.070, 5.0],
    "desensitization":     [1.00, 0.003, 0.025, 0.120, 0.30, 5.0],
    "coop_plus_linear":    [0.75, 0.004, 0.030, 2.0, 0.25, 0.100, 5.0],
    "diffusion_clearance": [1.00, 0.003, 0.015, 0.080, 0.65, 5.0],
    "double_cooperative":  [1.00, 0.003, 0.018, 2.0, 0.012, 0.090, 1.5, 5.0],
    "hetero_coop":         [1.00, 0.003, 0.020, 2.0, 0.65, 0.012, 0.090, 1.5, 5.0],
    "two_comp_coop":       [0.65, 0.003, 0.015, 2.0, 0.35, 0.012, 0.090, 1.5, 5.0],
}

ALL_MODELS = [name for names in MODEL_GROUPS.values() for name in names]
TIME_MS = np.linspace(-5, 140, 1200)

def model_curve(name, params=None):
    spec = get_event_model(name)
    values = EXAMPLE_PARAMS[spec["name"]] if params is None else params
    return spec, np.asarray(spec["func"](TIME_MS, *values), float)

def parameter_units(name):
    if name == "t_onset" or name in {"t_rise", "t_decay"}:
        return "ms"
    if "tau" in name:
        return "s"
    if name in {"kon", "koff"}:
        return "s⁻¹"
    if name.startswith("frac") or name in {"desens_factor"}:
        return "fraction"
    if name.startswith("n"):
        return "dimensionless"
    return "a.u."

for group, names in MODEL_GROUPS.items():
    print(f"\n{group}")
    print("-" * len(group))
    for name in names:
        spec = get_event_model(name)
        params = ", ".join(spec["params"])
        print(f"{spec['name']:<22} ({spec['complexity']} parameters): {params}")

---
## 2. Shape gallery

Every curve is normalized to its own maximum in this overview. This removes amplitude differences and highlights rise shape, peak timing, decay, shoulders, and long tails.

In [ ]:
fig, axes = plt.subplots(6, 3, figsize=(14, 18), sharex=True)
axes = axes.ravel()

for ax, name in zip(axes, ALL_MODELS):
    spec, y = model_curve(name)
    scale = np.nanmax(np.abs(y))
    y_norm = y / scale if np.isfinite(scale) and scale > 0 else y
    onset = EXAMPLE_PARAMS[spec["name"]][spec["params"].index("t_onset")]
    ax.plot(TIME_MS, y_norm, color="#245b78", lw=2)
    ax.axvline(onset, color="#b23a48", lw=0.9, ls="--", alpha=0.8)
    ax.axhline(0, color="0.8", lw=0.7)
    ax.set_title(f"{spec['name']}  |  {spec['complexity']} parameters", fontsize=10, weight="bold")
    ax.set_xlim(-3, 120)
    ax.set_ylim(-0.08, 1.12)
    ax.grid(True, alpha=0.18)
    ax.text(0.98, 0.84, "onset", transform=ax.transAxes, ha="right", color="#b23a48", fontsize=8)

for ax in axes[len(ALL_MODELS):]:
    ax.axis("off")
for ax in axes[-3:]:
    if ax.axison:
        ax.set_xlabel("Time (ms)")
for row in range(6):
    axes[row * 3].set_ylabel("Normalized response")

fig.suptitle("Available event-fitting models", fontsize=16, weight="bold", y=1.002)
finish_figure(fig, "01_all_model_shapes")

**Figure 1 | Shape gallery of all registered event-fitting models.** Responses are generated from representative parameter values and normalized to unit peak. Red dashed lines indicate event onset. Simple models primarily alter rise and decay curvature, whereas kinetic and composite models introduce shoulders, multiple decay regimes, desensitization, or persistent tails.

---
## 3. Inspect one model in detail

Change `MODEL_TO_INSPECT` to any canonical name printed in the catalogue. The figure labels observable curve features and displays the exact parameter vector, units, and active fit bounds from the registry.

In [ ]:
MODEL_TO_INSPECT = "iglusnfr_tri" # other options are 

def first_crossing(time, values, level, start_idx, direction):
    segment = values[start_idx:]
    if direction == "up":
        hits = np.flatnonzero(segment >= level)
    else:
        hits = np.flatnonzero(segment <= level)
    return time[start_idx + hits[0]] if hits.size else np.nan

spec, y = model_curve(MODEL_TO_INSPECT)
params = np.asarray(EXAMPLE_PARAMS[spec["name"]], float)
peak_idx = int(np.nanargmax(y))
peak_t, peak_y = TIME_MS[peak_idx], y[peak_idx]
onset_idx = spec["params"].index("t_onset")
onset = params[onset_idx]
onset_sample = int(np.argmin(np.abs(TIME_MS - onset)))
t10 = first_crossing(TIME_MS, y, 0.10 * peak_y, onset_sample, "up")
t90 = first_crossing(TIME_MS, y, 0.90 * peak_y, onset_sample, "up")
t_half = first_crossing(TIME_MS, y, 0.50 * peak_y, peak_idx, "down")

fig, (ax, ax_table) = plt.subplots(1, 2, figsize=(14, 5.5), gridspec_kw={"width_ratios": [1.45, 1]})
ax.plot(TIME_MS, y, color="#245b78", lw=2.5, label="model response")
ax.axvline(onset, color="#b23a48", ls="--", lw=1.2)
ax.annotate("event onset", xy=(onset, 0), xytext=(onset + 8, 0.18 * peak_y),
            arrowprops=dict(arrowstyle="->", color="#b23a48"), color="#b23a48")
ax.scatter([peak_t], [peak_y], color="#d1495b", s=55, zorder=5)
ax.annotate(f"peak\n({peak_t:.1f} ms)", xy=(peak_t, peak_y), xytext=(peak_t + 12, 0.88 * peak_y),
            arrowprops=dict(arrowstyle="->", color="#d1495b"), color="#8f2533")
if np.isfinite(t10) and np.isfinite(t90) and t90 > t10:
    ax.axvspan(t10, t90, color="#f4a261", alpha=0.22)
    ax.text((t10 + t90) / 2, 0.15 * peak_y, "10–90% rise", ha="center", color="#9b5b13")
if np.isfinite(t_half):
    ax.hlines(0.5 * peak_y, peak_t, t_half, color="#2a9d8f", lw=1.5)
    ax.vlines(t_half, 0, 0.5 * peak_y, color="#2a9d8f", lw=1, ls=":")
    ax.text((peak_t + t_half) / 2, 0.54 * peak_y, "post-peak half-decay", ha="center", color="#14756d")
ax.fill_between(TIME_MS, 0, y, where=TIME_MS >= peak_t + 25, color="#7a5195", alpha=0.12)
ax.text(0.72, 0.20, "late tail", transform=ax.transAxes, color="#684077")
ax.set(xlim=(-3, 120), xlabel="Time (ms)", ylabel="Model response (a.u.)",
       title=f"Labeled response features — {spec['name']}")
ax.grid(True, alpha=0.2)

ax_table.axis("off")
rows = []
lb, ub = spec["bounds"]
for name, value, lower, upper in zip(spec["params"], params, lb, ub):
    unit = parameter_units(name)
    value_text = f"{value:.4g}"
    bound_text = f"[{lower:.4g}, {upper:.4g}]" if np.isfinite(upper) else f"[{lower:.4g}, ∞)"
    rows.append([name, value_text, unit, bound_text])
table = ax_table.table(cellText=rows, colLabels=["Parameter", "Example", "Unit", "Fit bounds"],
                       cellLoc="left", colLoc="left", loc="center")
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1.05, 1.45)
for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_facecolor("#dceaf2")
        cell.set_text_props(weight="bold")
    else:
        cell.set_facecolor("#f7fafc" if row % 2 else "white")
    cell.set_edgecolor("#cad5dc")
ax_table.set_title("Model parameters", weight="bold", pad=12)

finish_figure(fig, f"02_labeled_{spec['name']}")

**Figure 2 | Labeled features and parameterization of one selected model.** The response panel identifies onset, 10–90% rise, peak, post-peak half-decay, and late tail. The adjacent table is populated directly from the model registry and distinguishes example values from admissible fitting bounds. Some parameters control a mechanistic rate or component mixture rather than a single geometric landmark; their effects are isolated below.

---
## 4. What does each parameter change?

Each panel varies one parameter while holding all others fixed. The bold curve is the representative value; lighter curves show a lower and higher value chosen within the active bounds. This is the most direct way to see whether a parameter controls amplitude, timing, rise cooperativity, decay, component balance, or the late tail.

In [ ]:
def useful_variation(value, lower, upper, name):
    """Choose readable low/base/high values without sitting on pathological bounds."""
    if name == "t_onset":
        return [max(lower, value - 2), value, min(upper, value + 2)]
    if name.startswith("frac") or name == "desens_factor":
        return [max(lower, value - 0.20), value, min(upper, value + 0.20)]
    if name.startswith("n"):
        return [max(lower, value * 0.6), value, min(upper, value * 1.7)]
    low = max(lower, value * 0.55)
    high = value * 1.8 if not np.isfinite(upper) else min(upper, value * 1.8)
    return [low, value, high]

n_params = len(spec["params"])
ncols = 3
nrows = int(np.ceil(n_params / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(14, 3.5 * nrows), sharex=True)
axes = np.atleast_1d(axes).ravel()
colors = ["#a8c5d5", "#245b78", "#d78c52"]
styles = ["--", "-", ":"]

lb, ub = spec["bounds"]
for idx, (ax, name, base, lower, upper) in enumerate(zip(axes, spec["params"], params, lb, ub)):
    varied_values = useful_variation(base, lower, upper, name)
    for value, color, style in zip(varied_values, colors, styles):
        trial = params.copy()
        trial[idx] = value
        y_trial = spec["func"](TIME_MS, *trial)
        label = f"{name}={value:.4g} {parameter_units(name)}"
        ax.plot(TIME_MS, y_trial, color=color, ls=style,
                lw=2.4 if value == base else 1.5, label=label)
    ax.set_title(name, weight="bold")
    ax.set_xlim(-3, 120)
    ax.set_xlabel("Time (ms)")
    ax.set_ylabel("Response")
    ax.legend(frameon=False, fontsize=7)
    ax.grid(True, alpha=0.18)

for ax in axes[n_params:]:
    ax.axis("off")

fig.suptitle(f"One-parameter-at-a-time sensitivity — {spec['name']}", fontsize=15, weight="bold", y=1.01)
finish_figure(fig, f"03_parameter_sensitivity_{spec['name']}")

**Figure 3 | One-parameter-at-a-time sensitivity of the selected model.** Each subplot isolates the visual consequence of changing one fitted parameter. The representative parameterization is shown as a bold solid curve, with lower and higher values shown as lighter dashed and dotted curves. Coupling between parameters remains possible during real fitting; these panels describe local interpretability, not statistical identifiability.

---
## 5. Component labels for the iGluSnFR models

For the bi- and tri-exponential iGluSnFR models, the decay is a weighted sum of kinetic components. The next figure labels those components explicitly; their fractions sum to one.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))

for ax, name in zip(axes, ["iglusnfr", "iglusnfr_tri"]):
    spec = get_event_model(name)
    p = EXAMPLE_PARAMS[name]
    y = spec["func"](TIME_MS, *p)
    onset = p[spec["params"].index("t_onset")]
    tp = np.maximum(TIME_MS - onset, 0) / 1000.0
    rise = 1 - np.exp(-tp / p[spec["params"].index("tau_rise")])
    rise[TIME_MS < onset] = 0

    if name == "iglusnfr":
        ff = p[spec["params"].index("frac_fast")]
        fs = 1 - ff
        components = [
            ("fast component", ff * rise * np.exp(-tp / p[spec["params"].index("tau_decay_fast")]), "tab:blue"),
            ("slow component", fs * rise * np.exp(-tp / p[spec["params"].index("tau_decay_slow")]), "tab:orange"),
        ]
    else:
        ff = p[spec["params"].index("frac_fast")]
        fs = p[spec["params"].index("frac_slow")]
        fss = max(0.0, 1 - ff - fs)
        components = [
            ("fast component", ff * rise * np.exp(-tp / p[spec["params"].index("tau_decay_fast")]), "tab:blue"),
            ("slow component", fs * rise * np.exp(-tp / p[spec["params"].index("tau_decay_slow")]), "tab:orange"),
            ("superslow component", fss * rise * np.exp(-tp / p[spec["params"].index("tau_decay_superslow")]), "tab:green"),
        ]

    total_components = np.sum([component for _, component, _ in components], axis=0)
    scale = np.nanmax(total_components)
    scale = scale if np.isfinite(scale) and scale > 0 else 1.0
    for label, component, color in components:
        component_norm = component / scale
        ax.plot(TIME_MS, component_norm, color=color, lw=1.8, label=label)
        ax.fill_between(TIME_MS, 0, component_norm, color=color, alpha=0.10)
    ax.plot(TIME_MS, total_components / scale, color="black", lw=2.4, label="normalized sum")
    ax.axvline(onset, color="#b23a48", ls="--", lw=1, label="onset")
    ax.set(xlim=(-3, 120), xlabel="Time (ms)", ylabel="Component contribution",
           title=f"{name}: labeled decay components")
    ax.legend(frameon=False, fontsize=8)
    ax.grid(True, alpha=0.18)

finish_figure(fig, "04_iglusnfr_components")

**Figure 4 | Component structure of the iGluSnFR models.** The bi-exponential model combines fast and slow decays, whereas the tri-exponential model adds a superslow component that captures persistent post-event fluorescence. Colored curves show weighted component contributions and the black curve shows their normalized sum. Fraction parameters control relative component weights; time constants control how quickly each contribution decays.

---
## Using the gallery

- Change `MODEL_TO_INSPECT` and rerun Sections 3–4 to inspect another registered model.
- Edit the corresponding `EXAMPLE_PARAMS` entry to explore a specific parameterization.
- Use `get_event_model(name)["bounds"]` as the authoritative current fitting bounds.
- Set `SAVE_FIGURES=True` to export PNG and SVG versions into `docs/model_fitting/figures/`.

For actual data fitting, use the calibration workflows in `Model_Calibration/` and the event-extraction pipeline in `Feature_extraction/extract_metrics.py`.